# 04 · Bi-LSTM with GloVe Embeddings
Loads DL text splits → builds vocabulary → loads GloVe → trains with a proper **train / val / test** loop including early stopping → evaluates on held-out test set.

**Prerequisite:** `00_data_preparation.ipynb`

> **Setup:** Download `glove.6B.100d.txt` from https://nlp.stanford.edu/data/glove.6B.zip and place it under `./glove.6B/`.

In [ ]:
import os, pickle, random, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from tqdm import tqdm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_DIR = '../outputs/featured/'
OUTPUT_DIR = '../outputs/bilstm/'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Device: {DEVICE}')

def load(name):
    with open(os.path.join(INPUT_DIR, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

X_dl_train = load('X_dl_train')
X_dl_val   = load('X_dl_val')
X_dl_test  = load('X_dl_test')
y_train    = load('y_train').to_numpy()
y_val      = load('y_val').to_numpy()
y_test     = load('y_test').to_numpy()

print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')


## 1 · Vocabulary (built from train only)

In [ ]:
MAX_VOCAB   = 30_000
MAX_SEQ_LEN = 512
EMBED_DIM   = 100
GLOVE_PATH  = '../glove.6B/glove.6B.100d.txt'

PAD_IDX, UNK_IDX = 0, 1

all_tokens  = [t for text in X_dl_train for t in text.split()]
counter     = Counter(all_tokens)
vocab_words = [w for w, _ in counter.most_common(MAX_VOCAB)]

word2idx             = {w: i+2 for i, w in enumerate(vocab_words)}
word2idx['<PAD>']   = PAD_IDX
word2idx['<UNK>']   = UNK_IDX
VOCAB_SIZE = len(word2idx)
print(f'Vocabulary size: {VOCAB_SIZE:,}')


## 2 · Load GloVe Embeddings

In [ ]:
def load_glove(path, word2idx, embed_dim=100):
    embedding_matrix         = np.random.uniform(-0.1, 0.1, (len(word2idx), embed_dim))
    embedding_matrix[PAD_IDX] = 0.0
    found = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc='Loading GloVe'):
            parts = line.strip().split()
            word  = parts[0]
            if word in word2idx:
                embedding_matrix[word2idx[word]] = np.array(parts[1:], dtype=float)
                found += 1
    print(f'GloVe words found in vocab: {found:,} / {len(word2idx):,}')
    return torch.tensor(embedding_matrix, dtype=torch.float32)

embedding_matrix = load_glove(GLOVE_PATH, word2idx, EMBED_DIM)


## 3 · Dataset & DataLoaders

In [ ]:
def encode_text(text, word2idx, max_len):
    tokens = text.split()[:max_len]
    return [word2idx.get(t, UNK_IDX) for t in tokens]

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.data   = [torch.tensor(encode_text(t, word2idx, MAX_SEQ_LEN), dtype=torch.long)
                       for t in tqdm(texts, desc='Encoding')]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):           return len(self.labels)
    def __getitem__(self, i):    return self.data[i], self.labels[i]

def collate_fn(batch):
    texts, labels = zip(*batch)
    return pad_sequence(texts, batch_first=True, padding_value=PAD_IDX), torch.stack(labels)

BATCH_SIZE = 64

train_ds     = TextDataset(X_dl_train.tolist(), y_train)
val_ds       = TextDataset(X_dl_val.tolist(),   y_val)
test_ds      = TextDataset(X_dl_test.tolist(),  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print('DataLoaders ready.')


## 4 · Bi-LSTM Architecture

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers,
                 num_classes, dropout, pretrained_embeddings):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.embedding.weight = nn.Parameter(pretrained_embeddings, requires_grad=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb          = self.dropout(self.embedding(x))
        out, _       = self.lstm(emb)
        attn_weights = torch.softmax(self.attention(out), dim=1)
        context      = (attn_weights * out).sum(dim=1)
        return self.fc(self.dropout(context))

model = BiLSTMClassifier(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
    hidden_dim=256, num_layers=2, num_classes=2,
    dropout=0.3, pretrained_embeddings=embedding_matrix
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')


## 5 · Training with Early Stopping on Validation F1

In [ ]:
LSTM_EPOCHS   = 10          # max epochs; early stopping will likely terminate sooner
LSTM_LR       = 1e-3
PATIENCE      = 3           # stop if val F1 does not improve for this many epochs

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=1, factor=0.5, verbose=True)

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for texts, labels in tqdm(loader, desc='  Train' if train else '  Eval '):
            texts, labels = texts.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(texts)
            loss   = criterion(logits, labels)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / total
    acc      = correct / total
    f1       = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1, np.array(all_preds)

history        = []
best_val_f1    = -1.0
epochs_no_impr = 0
best_state     = None

for epoch in range(1, LSTM_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{LSTM_EPOCHS}')
    tr_loss, tr_acc, tr_f1, _ = run_epoch(model, train_loader, train=True)
    val_loss, val_acc, val_f1, _ = run_epoch(model, val_loader, train=False)
    scheduler.step(val_loss)

    history.append(dict(epoch=epoch,
                        tr_loss=tr_loss, tr_acc=tr_acc, tr_f1=tr_f1,
                        val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))

    print(f'  Train  loss={tr_loss:.4f}  acc={tr_acc:.4f}  f1={tr_f1:.4f}')
    print(f'  Val    loss={val_loss:.4f}  acc={val_acc:.4f}  f1={val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1    = val_f1
        epochs_no_impr = 0
        best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  ✓ New best val F1: {best_val_f1:.4f}')
    else:
        epochs_no_impr += 1
        print(f'  No improvement ({epochs_no_impr}/{PATIENCE})')
        if epochs_no_impr >= PATIENCE:
            print('Early stopping triggered.')
            break

# Restore best weights
model.load_state_dict(best_state)
print(f'\nTraining complete. Best Val F1: {best_val_f1:.4f}')


## 6 · Evaluate on Test Set

In [ ]:
_, _, _, y_pred_lstm = run_epoch(model, test_loader, train=False)

metrics = dict(
    Accuracy  = accuracy_score(y_test, y_pred_lstm),
    Precision = precision_score(y_test, y_pred_lstm, average='weighted'),
    Recall    = recall_score(y_test, y_pred_lstm, average='weighted'),
    F1        = f1_score(y_test, y_pred_lstm, average='weighted'),
)
print('\n=== Bi-LSTM (Test) ===')
for k, v in metrics.items():
    print(f'  {k:10s}: {v:.4f}')
print()
print(classification_report(y_test, y_pred_lstm, target_names=['Fake', 'Real']))

cm = confusion_matrix(y_test, y_pred_lstm)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake','Real'], yticklabels=['Fake','Real'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Bi-LSTM')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_Bi-LSTM.png'), dpi=120)
plt.show()


## 7 · Training Curve

In [ ]:
hist_df = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_df['epoch'], hist_df['tr_loss'],  label='Train'); ax1.plot(hist_df['epoch'], hist_df['val_loss'],  label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(hist_df['epoch'], hist_df['tr_f1'],  label='Train'); ax2.plot(hist_df['epoch'], hist_df['val_f1'],  label='Val')
ax2.set_title('F1'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.suptitle('Bi-LSTM Training Curve', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'curve_bilstm.png'), dpi=120)
plt.show()


## 8 · Save Artefacts

In [ ]:
torch.save(best_state, os.path.join(OUTPUT_DIR, 'model_bilstm.pt'))

with open(os.path.join(OUTPUT_DIR, 'word2idx.pkl'), 'wb') as f:
    pickle.dump(word2idx, f)

with open(os.path.join(OUTPUT_DIR, 'pred_bilstm.pkl'), 'wb') as f:
    pickle.dump(y_pred_lstm, f)

with open(os.path.join(OUTPUT_DIR, 'metrics_bilstm.json'), 'w') as f:
    json.dump({'test': metrics, 'history': history}, f, indent=2)

print('Saved: model_bilstm.pt, word2idx.pkl, pred_bilstm.pkl, metrics_bilstm.json')
